In [1]:
import pyspark
from pyspark.sql import SparkSession

from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, PCA, Imputer
from pyspark.ml.classification import OneVsRest
from pyspark.ml import Pipeline
from pyspark.sql.functions import mean, col, expr
import numpy as np
import time

In [ ]:
spark.sparkContext.stop()

In [2]:
spark = SparkSession.builder.appName("ce53") \
    .config("SPARK_LOCAL_IP", "192.168.1.2") \
    .master("spark://192.168.1.2:7077") \
    .config("spark.driver.cores", "2") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "20g") \
    .config("spark.executor.cores", "16") \
    .config("spark.executor.instances", "6") \
    .config("spark.shuffle.partitions", "6") \
    .config("spark.kryoserializer.buffer.max", "256m") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.executor.heartbeatInterval","11999s") \
    .config("spark.network.timeout","12000s") \
.getOrCreate()

24/05/12 21:21:58 WARN Utils: Your hostname, ubuntu-virtual-machine resolves to a loopback address: 127.0.1.1; using 192.168.1.106 instead (on interface ens33)
24/05/12 21:21:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/05/12 21:21:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
parquet_files = ["hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2021-12-12 - 2021-12-19/part-00000-7c2e9adb-5430-4792-a42b-10ff5bbd46e8-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2021-12-19 - 2021-12-26/part-00000-3f86626a-1225-47f9-a5a2-0170b737e404-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2021-12-26 - 2022-01-02/part-00000-b1a9fc13-8068-4a5d-91b2-871438709e81-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2022-01-02 - 2022-01-09/part-00000-26e9208e-7819-451b-b23f-2e47f6d1e834-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2022-01-09 - 2022-01-16/part-00000-36240b61-b84f-4164-a873-d7973e652780-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2022-01-16 - 2022-01-23/part-00000-cbf26680-106d-40e7-8278-60520afdbb0e-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2022-02-06 - 2022-02-13/part-00000-df678a79-4a73-452b-8e72-d624b2732f17-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2022-02-13 - 2022-02-20/part-00000-1da06990-329c-4e38-913a-0f0aa39b388d-c000.snappy.parquet"]

In [4]:
#Read the parquet files
df = spark.read.parquet(*parquet_files, inferSchema=True)

In [5]:
#Get unique label counts
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

label_counts.show()

+--------------------+-------+
|        label_tactic|  count|
+--------------------+-------+
|   Credential Access|     31|
|     Defense Evasion|      1|
|           Discovery|   2086|
|        Exfiltration|      7|
|      Initial Access|      1|
|    Lateral Movement|      4|
|         Persistence|      1|
|Privilege Escalation|     13|
|      Reconnaissance|9278722|
|Resource Development|      3|
|                none|9281599|
+--------------------+-------+



In [6]:
start_time = time.time()

#Drop labels and get remaining counts
labels_to_drop = ['Defense Evasion', 
                  'Exfiltration',
                  'Initial Access',
                  'Lateral Movement', 
                  'Persistence',
                  'Privilege Escalation', 
                  'Resource Development', 
                  'Credential Access', 
                  'Discovery']

df = df.filter(~col("label_tactic").isin(labels_to_drop))
filtered_label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

filtered_label_counts.show()

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

+--------------+-------+
|  label_tactic|  count|
+--------------+-------+
|Reconnaissance|9278722|
|          none|9281599|
+--------------+-------+

Execution time: 2.61637282371521 seconds


In [7]:
#Drop uid feature
df = df.drop('uid')

In [8]:
start_time = time.time()

#Columns to index
columns_to_index = ['service', 
                    'conn_state', 
                    'history', 
                    'proto', 
                    'dest_ip_zeek', 
                    'community_id', 
                    'src_ip_zeek',
                    'datetime',
                    'local_resp',
                    'local_orig',
                    'label_tactic']

#Cast datetime, local_resp, local_orig to String
df = df.withColumn("datetime", col("datetime").cast("string"))
df = df.withColumn("local_resp", col("local_resp").cast("string"))
df = df.withColumn("local_orig", col("local_orig").cast("string"))

#Impute null values with empty string
for column in columns_to_index:
    df = df.fillna('', subset=[column])

In [9]:
#Split the into training and test sets
start_time = time.time()
train_data, test_data = df.randomSplit([0.7, 0.3], seed=42)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.026652097702026367 seconds


In [10]:
#StringIndexer
indexers = [StringIndexer(inputCol=column, outputCol=column+"_indexed").setHandleInvalid("keep") for column in columns_to_index]

#Chain indexers together
pipeline = Pipeline(stages=indexers).fit(train_data)

#Fit and transform the data
train_data_indexed = pipeline.transform(train_data)
test_data_indexed = pipeline.transform(test_data)

#Drop original columns
train_data_indexed = train_data_indexed.drop(*columns_to_index)
train_data_indexed = train_data_indexed.withColumnRenamed("label_tactic_indexed", "label_tactic")
test_data_indexed = test_data_indexed.drop(*columns_to_index)
test_data_indexed = test_data_indexed.withColumnRenamed("label_tactic_indexed", "label_tactic")

print("train_indexed columns: ", train_data_indexed.columns)
print("test_indexed columns: ", test_data_indexed.columns)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

train_indexed columns:  ['resp_pkts', 'orig_ip_bytes', 'missed_bytes', 'duration', 'orig_pkts', 'resp_ip_bytes', 'dest_port_zeek', 'orig_bytes', 'resp_bytes', 'src_port_zeek', 'ts', 'service_indexed', 'conn_state_indexed', 'history_indexed', 'proto_indexed', 'dest_ip_zeek_indexed', 'community_id_indexed', 'src_ip_zeek_indexed', 'datetime_indexed', 'local_resp_indexed', 'local_orig_indexed', 'label_tactic']
test_indexed columns:  ['resp_pkts', 'orig_ip_bytes', 'missed_bytes', 'duration', 'orig_pkts', 'resp_ip_bytes', 'dest_port_zeek', 'orig_bytes', 'resp_bytes', 'src_port_zeek', 'ts', 'service_indexed', 'conn_state_indexed', 'history_indexed', 'proto_indexed', 'dest_ip_zeek_indexed', 'community_id_indexed', 'src_ip_zeek_indexed', 'datetime_indexed', 'local_resp_indexed', 'local_orig_indexed', 'label_tactic']
Execution time: 251.89830017089844 seconds


In [11]:
start_time = time.time()

#List of numeric column names
numeric_columns = ['resp_pkts', 
                   'orig_ip_bytes', 
                   'missed_bytes', 
                   'duration', 
                   'orig_pkts',
                   'resp_ip_bytes', 
                   'dest_port_zeek', 
                   'orig_bytes', 
                   'resp_bytes',
                   'src_port_zeek', 
                   'ts']

#Create Imputer
imputer = Imputer(
    inputCols=numeric_columns,
    outputCols=["{}_imputed".format(column) for column in numeric_columns]
)

#Fit the imputer to the training data
imputer_model = imputer.setStrategy("mean").fit(train_data_indexed)

#Apply the Imputer to the training data
train_data_imputed = imputer_model.transform(train_data_indexed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Apply the Imputer to the test data
start_time = time.time()
test_data_imputed = imputer_model.transform(test_data_indexed)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

train_data_imputed = train_data_imputed.drop(*numeric_columns)
test_data_imputed = test_data_imputed.drop(*numeric_columns)

print("\nTrain data imputed: ", train_data_imputed.columns)
print("\n")
print("Test data imputed: ", test_data_imputed.columns)

Execution time: 20.601871252059937 seconds
Execution time: 0.029946565628051758 seconds

Train data imputed:  ['service_indexed', 'conn_state_indexed', 'history_indexed', 'proto_indexed', 'dest_ip_zeek_indexed', 'community_id_indexed', 'src_ip_zeek_indexed', 'datetime_indexed', 'local_resp_indexed', 'local_orig_indexed', 'label_tactic', 'resp_pkts_imputed', 'orig_ip_bytes_imputed', 'missed_bytes_imputed', 'duration_imputed', 'orig_pkts_imputed', 'resp_ip_bytes_imputed', 'dest_port_zeek_imputed', 'orig_bytes_imputed', 'resp_bytes_imputed', 'src_port_zeek_imputed', 'ts_imputed']


Test data imputed:  ['service_indexed', 'conn_state_indexed', 'history_indexed', 'proto_indexed', 'dest_ip_zeek_indexed', 'community_id_indexed', 'src_ip_zeek_indexed', 'datetime_indexed', 'local_resp_indexed', 'local_orig_indexed', 'label_tactic', 'resp_pkts_imputed', 'orig_ip_bytes_imputed', 'missed_bytes_imputed', 'duration_imputed', 'orig_pkts_imputed', 'resp_ip_bytes_imputed', 'dest_port_zeek_imputed', 'or

In [12]:
start_time = time.time()

#Create VectorAssembler
columns_to_assemble = [column for column in train_data_imputed.columns if column.endswith("_imputed") or column.endswith("_indexed")]
print("Columns to assemble: ", columns_to_assemble)

assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

#Transform the training data
train_data_assembled = assembler.transform(train_data_imputed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Transform the test data
start_time = time.time()
test_data_assembled = assembler.transform(test_data_imputed)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Select the features and label columns
train_data_assembled = train_data_assembled.select("features", "label_tactic")
test_data_assembled = test_data_assembled.select("features", "label_tactic")

print("Train_data_assembled columns: ", train_data_assembled.columns)
print("Test_data_assembled columns: ", test_data_assembled.columns)

#filtered_label_counts = train_data_assembled.groupBy("label_tactic").count()
#filtered_label_counts.show()
#filtered_label_counts = train_data_assembled.groupBy("features").count()
#filtered_label_counts.limit(20).show()

Columns to assemble:  ['service_indexed', 'conn_state_indexed', 'history_indexed', 'proto_indexed', 'dest_ip_zeek_indexed', 'community_id_indexed', 'src_ip_zeek_indexed', 'datetime_indexed', 'local_resp_indexed', 'local_orig_indexed', 'resp_pkts_imputed', 'orig_ip_bytes_imputed', 'missed_bytes_imputed', 'duration_imputed', 'orig_pkts_imputed', 'resp_ip_bytes_imputed', 'dest_port_zeek_imputed', 'orig_bytes_imputed', 'resp_bytes_imputed', 'src_port_zeek_imputed', 'ts_imputed']
Execution time: 2.3679988384246826 seconds
Execution time: 1.9404997825622559 seconds
Train_data_assembled columns:  ['features', 'label_tactic']
Test_data_assembled columns:  ['features', 'label_tactic']


In [13]:
#Create the SVM model
start_time = time.time()
svm = LinearSVC(labelCol="label_tactic", featuresCol="features", maxIter=10, regParam=0.0, tol=.00001, fitIntercept=True)
ovr = OneVsRest(classifier=svm, labelCol="label_tactic")
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Fit the model
start_time = time.time()
svm_model = ovr.fit(train_data_assembled)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.1074526309967041 seconds


24/05/12 21:27:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
24/05/12 21:27:31 WARN DAGScheduler: Broadcasting large task binary with size 86.2 MiB
24/05/12 21:28:20 WARN DAGScheduler: Broadcasting large task binary with size 86.2 MiB
24/05/12 21:28:34 WARN DAGScheduler: Broadcasting large task binary with size 86.2 MiB
24/05/12 21:28:45 WARN DAGScheduler: Broadcasting large task binary with size 86.2 MiB
24/05/12 21:29:05 WARN DAGScheduler: Broadcasting large task binary with size 86.2 MiB
ERROR:root:Exception while sending command.                         (0 + 9) / 9]
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above 

Py4JError: An error occurred while calling o1216.fit

In [ ]:
#Make predictions
start_time = time.time()

predictions = svm_model.transform(test_data_assembled)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

In [ ]:
# Evaluate the model
# Calculate accuracy
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="accuracy")
accuracy = evaluator_accuracy.evaluate(predictions)

# Calculate precision
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedPrecision")
precision = evaluator_precision.evaluate(predictions)

# Calculate recall
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedRecall")
recall = evaluator_recall.evaluate(predictions)

# Calculate F1-score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)

#Calculate FPR
evaluator_fprL = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="falsePositiveRateByLabel")
fprL_score = evaluator_fprL.evaluate(predictions)

#Calculate Weighted FPR
evaluator_fpr = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedFalsePositiveRate")
fpr_score = evaluator_fpr.evaluate(predictions)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1_score)
print("FPR by Label:", fprL_score)
print("Weighted FPR:", fpr_score)

In [ ]:
start_time = time.time()

#Extract predictions and labels
predictions_and_labels = predictions.select("prediction", "label_tactic")

#Calculate false positives and true negatives
false_positives = predictions_and_labels.filter((predictions_and_labels.prediction == 1) & (predictions_and_labels.label_tactic== 0)).count()
true_negatives = predictions_and_labels.filter((predictions_and_labels.prediction == 0) & (predictions_and_labels.label_tactic == 0)).count()

#Calculate FPR
fpr = false_positives / (false_positives + true_negatives)

print("False Positive Rate:", fpr)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

In [ ]:
from pyspark.mllib.evaluation import BinaryClassificationMetrics
from pyspark.sql import Row

# Convert DataFrame to RDD of tuples (prediction, label)
prediction_and_labels = predictions.select("prediction", "label_tactic") \
    .rdd.map(lambda row: (float(row['prediction']), float(row['label_tactic'])))


# Instantiate BinaryClassificationMetrics
metrics = BinaryClassificationMetrics(prediction_and_labels)

# Compute AUROC
auROC = metrics.areaUnderROC

# Print AUROC
print("Area under ROC = ", auROC)

In [ ]:
spark.sparkContext.stop()